# Slides to Markdown (no vision LLM)

Converts PDF lecture slides into clean, structured Markdown for LLM ingestion. Upload a few PDFs (or a `.zip` of them) and download one ZIP back.

**How it works.** The notebook reads the PDF's own text layer with [PyMuPDF](https://pymupdf.readthedocs.io), so it is fast and exact. No vision LLM is involved.
- **Text and layout.** Slide titles, nested bullets, numbered lists, two-column layouts, bold text, links and tables (ruled or borderless) are rebuilt from font sizes and positions.
- **Repeated content.** Headers, footers, slide numbers and template logos that repeat on every slide are removed.
- **Invisible OCR layers.** If the PDF was OCR'd, that layer is not used for normal text, which avoids duplicates. It is only used to recover text inside pictures, such as screenshots of questions.
- **Figures.** Pictures and vector diagrams are saved as PNG files. The Markdown links to each one and adds a caption placeholder plus the labels found on the figure.
- **Math.** Formulas are written as LaTeX (`$...$`, `$$...$$`) from the native math fonts. [pix2tex](https://github.com/lukas-blecher/LaTeX-OCR), a small formula-OCR model (not an LLM), is only used for 2-D formulas such as fractions and sums, and its output is checked against the PDF's own symbols.
- **Build slides.** Incremental "build" slides (the same slide revealed step by step) are collapsed into the final step.

**Output layout** (inside `slides_markdown.zip`):
```
<name>.md                 one file: YAML front-matter, then every PDF's "## Slide N: Title" sections
images/<deck>_slideNN_figK.png
```

Run the cells top to bottom. A GPU is **not** needed; pix2tex runs fine on CPU for a few formulas per deck.

## 1. Install dependencies

In [ ]:
#@title Install PyMuPDF and pix2tex (~1-2 min)
# pix2tex 0.1.4 needs the older albumentations API.
!pip install -q "pymupdf==1.28.2" "pix2tex==0.1.4" "albumentations==1.3.1" pillow
import pymupdf
print("PyMuPDF", pymupdf.VersionBind)

## 2. Options

- **MATH_ENGINE**
  - `auto` (recommended): exact LaTeX from the PDF's math fonts, plus pix2tex for 2-D formulas only.
  - `pix2tex`: tries pix2tex on every formula, but still keeps the font-based LaTeX when pix2tex disagrees with the PDF's symbols.
  - `rules`: never loads pix2tex.
- **DROP_BUILD_SLIDES**: keep only the last step of incremental build slides.
- **USE_OCR_LAYER**: recover text from images through the PDF's hidden OCR layer, if it has one.
- **DETECT_TABLES**: rebuild tables as Markdown tables.
- **INCLUDE_FIGURE_LABELS**: list the text found on each figure (axis labels, legends) under it.
- **FIGURE_DPI**: resolution of the exported PNGs.
- **MIN_FIGURE_AREA**: figures smaller than this fraction of the slide (icons, bullets) are ignored.
- **OUTPUT_NAME**: name of the single Markdown file, without `.md` (for example `lecture1_intro`). Empty uses the shared start of the PDF filenames.
- **DRIVE_INPUT_FOLDER**: optional. A Google Drive folder with PDFs (for example `/content/drive/MyDrive/course/week_01/slides`). It is used instead of uploading.
- **DRIVE_OUTPUT_FOLDER**: optional. Results are also copied there, for example into an Obsidian vault on Drive.

In [ ]:
#@title Conversion options
MATH_ENGINE = "auto"  #@param ["auto", "pix2tex", "rules"]
DROP_BUILD_SLIDES = True  #@param {type:"boolean"}
USE_OCR_LAYER = True  #@param {type:"boolean"}
DETECT_TABLES = True  #@param {type:"boolean"}
INCLUDE_FIGURE_LABELS = True  #@param {type:"boolean"}
FIGURE_DPI = 200  #@param {type:"slider", min:72, max:400, step:4}
MIN_FIGURE_AREA = 0.01  #@param {type:"number"}
OUTPUT_NAME = ""  #@param {type:"string"}
DRIVE_INPUT_FOLDER = ""  #@param {type:"string"}
DRIVE_OUTPUT_FOLDER = ""  #@param {type:"string"}

USE_PIX2TEX = MATH_ENGINE != "rules"
OPTS = {
    "use_pix2tex": USE_PIX2TEX,
    "math_engine": MATH_ENGINE,
    "drop_build_slides": DROP_BUILD_SLIDES,
    "use_ocr_layer": USE_OCR_LAYER,
    "detect_tables": DETECT_TABLES,
    "include_figure_labels": INCLUDE_FIGURE_LABELS,
    "figure_dpi": int(FIGURE_DPI),
    "min_figure_area": float(MIN_FIGURE_AREA),
}
print(OPTS)

## 3. Load the formula model (skipped when `MATH_ENGINE = rules`)

In [ ]:
#@title Load pix2tex once
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
latex_ocr = None
if USE_PIX2TEX:
    import torch
    from pix2tex.cli import LatexOCR
    latex_ocr = LatexOCR()
    print("pix2tex loaded on", "GPU" if torch.cuda.is_available() else "CPU")
else:
    print("pix2tex disabled; formulas use the font-based converter only")

## 4. Converter

All the conversion logic lives in this cell. Run it once; nothing visible happens.

In [ ]:
#@title Converter functions (run once)
import datetime, json, re
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path

import pymupdf

LIGATURES = {"\ufb00": "ff", "\ufb01": "fi", "\ufb02": "fl", "\ufb03": "ffi", "\ufb04": "ffl",
             "\ufb05": "st", "\ufb06": "st", "\u00a0": " ", "\u00ad": ""}
BULLET_GLYPHS = set("•●○◦▪▫■□►▸▹‣⁃∙·➢➤❖✓✔◆◇") | {"\uf0a7", "\uf0b7", "\uf0d8", "\uf0fc", "\uf076", "\uf06e", "\uf0a8"}
DASH_BULLETS = {"-", "–", "—", "*", "→"}
NUMBERED_RE = re.compile(r"^(\(?\d{1,2}[.)]|\(?[a-zA-Z][.)])\s")
MATH_CORE_RE = re.compile(r"^(CMMI|CMSY|CMEX|CMBSY|CMMIB|MSAM|MSBM|EUFM|EUSM|EUEX|RSFS|STIX|XITS|Asana|Symbol|MT-?Extra|Euclid|"
                          r"rtxmi|txmi|txsy|pxmi|pxsy|ntxmi|ntxsy|zxlr|MathematicalPi)|Math", re.I)
MATH_COMPAT_RE = re.compile(r"^(CM[A-Z]{1,4}\d|SF[A-Z]{2}\d|LM|lmr|cmr|EC|TS|rtxr|txr|pxr)", re.I)
MATH_OP_CHARS = set("∑∏∫∂∇√≤≥≠≈∈∉⊂⊆∪∩∀∃∞±∓×÷∝≡")
FUNC_NAMES = {"sin", "cos", "tan", "log", "ln", "exp", "max", "min", "arg", "det", "lim", "sup", "inf",
              "argmax", "argmin", "tanh", "sigmoid", "softmax", "Pr", "var", "cov", "tr", "sign", "diag"}
GREEK = dict(zip("αβγδεζηθικλμνξπρστυφχψωϵϑϕϱςΓΔΘΛΞΠΣΥΦΨΩ",
                 ["alpha", "beta", "gamma", "delta", "epsilon", "zeta", "eta", "theta", "iota", "kappa", "lambda",
                  "mu", "nu", "xi", "pi", "rho", "sigma", "tau", "upsilon", "phi", "chi", "psi", "omega",
                  "epsilon", "vartheta", "varphi", "varrho", "varsigma", "Gamma", "Delta", "Theta", "Lambda", "Xi",
                  "Pi", "Sigma", "Upsilon", "Phi", "Psi", "Omega"]))
LATEX_CHARS = {"→": r"\to", "←": r"\leftarrow", "⇒": r"\Rightarrow", "⇔": r"\Leftrightarrow", "↦": r"\mapsto",
               "∈": r"\in", "∉": r"\notin", "≤": r"\leq", "≥": r"\geq", "≠": r"\neq", "≈": r"\approx",
               "∼": r"\sim", "∝": r"\propto", "×": r"\times", "·": r"\cdot", "⋅": r"\cdot", "∗": "*", "−": "-",
               "±": r"\pm", "∓": r"\mp", "∞": r"\infty", "∑": r"\sum", "∏": r"\prod", "∫": r"\int",
               "∂": r"\partial", "∇": r"\nabla", "√": r"\sqrt", "⊂": r"\subset", "⊆": r"\subseteq", "∪": r"\cup",
               "∩": r"\cap", "∀": r"\forall", "∃": r"\exists", "¬": r"\neg", "∧": r"\wedge", "∨": r"\vee",
               "′": "'", "‖": r"\|", "{": r"\{", "}": r"\}", "…": r"\ldots", "⋯": r"\cdots", "∘": r"\circ",
               "⊤": r"\top", "†": r"\dagger", "ℓ": r"\ell", "∅": r"\emptyset", "≡": r"\equiv", "≪": r"\ll",
               "≫": r"\gg", "⟨": r"\langle", "⟩": r"\rangle", "%": r"\%", "#": r"\#", "&": r"\&", "÷": r"\div",
               "~": r"\sim", "\ufffd": "?", "\u0338": r"\not", "ˆ": r"\hat{}", "˜": r"\tilde{}"}


# Greek letters from TeX math-italic fonts that some PDF producers map to the wrong Unicode code points.
CMMI_RE = re.compile(r"^(CMMI|LMMathItalic|rtxmi|txmi|pxmi)", re.I)
CMMI_FIXES = {"↵": "α", "⇡": "π", "✓": "θ", "⌧": "τ", "✏": "ε", "⇢": "ρ", "⌘": "κ"}
CMSY_RE = re.compile(r"^(CMSY|CMBSY|LMMathSymbols|rtxsy|txsy|pxsy)", re.I)
CMSY_FIXES = {"⇤": "∗", "⇥": "×", "6": "\u0338"}


@dataclass
class Span:
    text: str
    x0: float
    y0: float
    x1: float
    y1: float
    size: float
    font: str
    flags: int
    oy: float
    ocr: bool = False
    rotated: bool = False
    lead: bool = False
    trail: bool = False

    @property
    def cx(self):
        return (self.x0 + self.x1) / 2

    @property
    def cy(self):
        return (self.y0 + self.y1) / 2

    @property
    def bold(self):
        return bool(self.flags & 16) or bool(re.search(r"bold|black|heavy|BX\d", self.font, re.I))

    @property
    def core(self):
        return self.oy - 0.7 * self.size, self.oy + 0.05 * self.size

    @property
    def rect(self):
        return pymupdf.Rect(self.x0, self.y0, self.x1, self.y1)


@dataclass
class Segment:
    spans: list
    bullet: str = None
    bullet_x: float = None
    row: int = -1

    @property
    def x0(self):
        return min(s.x0 for s in self.spans)

    @property
    def x1(self):
        return max(s.x1 for s in self.spans)

    @property
    def y0(self):
        return min(s.y0 for s in self.spans)

    @property
    def y1(self):
        return max(s.y1 for s in self.spans)

    @property
    def size(self):
        return max(s.size for s in self.spans)

    @property
    def rect(self):
        return pymupdf.Rect(self.x0, self.y0, self.x1, self.y1)

    @property
    def cx(self):
        return (self.x0 + self.x1) / 2

    @property
    def cy(self):
        return (self.y0 + self.y1) / 2

    @property
    def plain(self):
        return join_plain(self.spans)

    @property
    def content_spans(self):
        return self.spans[1:] if self.bullet == "glyph" else self.spans

    @property
    def text_x(self):
        cs = self.content_spans
        if self.bullet == "number":
            m = NUMBERED_RE.match(self.plain)
            acc = ""
            for s in cs:
                if len(acc.replace(" ", "")) >= len(m.group(0).replace(" ", "")):
                    return s.x0
                acc += s.text
        return min(s.x0 for s in cs) if cs else self.x0


@dataclass
class Stats:
    formulas_inline: int = 0
    formulas_display: int = 0
    pix2tex_used: int = 0
    pix2tex_rejected: int = 0
    complex_formulas: int = 0
    figures: int = 0
    tables: int = 0
    ocr_image_texts: int = 0
    warnings: list = field(default_factory=list)
    math_log: list = field(default_factory=list)


# ---------------------------------------------------------------- extraction

def base_font(name):
    return name.split("+", 1)[-1]


def clean_text(t):
    for k, v in LIGATURES.items():
        t = t.replace(k, v)
    return t


def chars_to_text(chars, size, font=""):
    out, prev = [], None
    cmsy = bool(CMSY_RE.match(font))
    for c in chars:
        ch = c["c"]
        if cmsy and ch == "→" and (c["bbox"][2] - c["bbox"][0]) < 0.95 * size:
            ch = "−"
        if prev is not None and not ch.isspace() and not prev["c"].isspace():
            if c["bbox"][0] - prev["bbox"][2] > 0.1 * size:
                out.append(" ")
        out.append(ch)
        prev = c
    return clean_text("".join(out))


def extract_spans(page):
    spans = []
    d = page.get_text("rawdict")
    for b in d["blocks"]:
        if b.get("type") != 0:
            continue
        for line in b["lines"]:
            dx, dy = line["dir"]
            rotated = abs(dy) > 0.1 or dx < 0
            for s in line["spans"]:
                font = base_font(s["font"])
                raw = chars_to_text(s["chars"], s["size"], font)
                if not raw.strip():
                    continue
                ocr = s.get("alpha", 255) == 0 or "GlyphLess" in font
                x0, y0, x1, y1 = s["bbox"]
                text = raw.strip()
                if CMMI_RE.match(font):
                    text = "".join(CMMI_FIXES.get(ch, ch) for ch in text)
                elif CMSY_RE.match(font):
                    text = "".join(CMSY_FIXES.get(ch, ch) for ch in text)
                spans.append(Span(text, x0, y0, x1, y1, s["size"], font, s["flags"], s["origin"][1],
                                  ocr=ocr, rotated=rotated, lead=raw[:1].isspace(), trail=raw[-1:].isspace()))
    return dedupe_spans(spans)


# ---------------------------------------------------------------- rows and segments

def build_rows(spans):
    rows = []
    for s in sorted(spans, key=lambda s: -s.size):
        a0, a1 = s.core
        best, best_ov = None, 0.0
        for r in rows:
            b0, b1 = r["core"]
            ov = min(a1, b1) - max(a0, b0)
            frac = ov / max(min(a1 - a0, b1 - b0), 0.1)
            gap = min(max(o.x0 - s.x1, s.x0 - o.x1, 0) for o in r["spans"])
            if gap > 3 * s.size:
                frac -= 0.4
            if frac > best_ov:
                best, best_ov = r, frac
        if best is not None and best_ov >= 0.4:
            best["spans"].append(s)
        else:
            rows.append({"core": (a0, a1), "spans": [s]})
    for r in rows:
        r["spans"].sort(key=lambda s: (s.x0, s.y0))
    rows.sort(key=lambda r: r["core"][0])
    return [r["spans"] for r in rows]


def split_segments(row):
    segs, cur = [], [row[0]]
    for s in row[1:]:
        prev_x1 = max(p.x1 for p in cur)
        size = max(min(s.size, cur[-1].size), 4)
        if s.x0 - prev_x1 > max(1.3 * size, 6):
            segs.append(cur)
            cur = [s]
        else:
            cur.append(s)
    segs.append(cur)
    return [Segment(c) for c in segs]


def make_segments(spans):
    segs = []
    for k, row in enumerate(build_rows(spans)):
        for seg in split_segments(row):
            seg.row = k
            segs.append(seg)
    return segs


def dedupe_spans(spans):
    """Drops text drawn twice at (almost) the same position, e.g. fake bold or overlays."""
    by_text, keep = {}, []
    for s in spans:
        dup = False
        for o in by_text.get((s.text, s.ocr), []):
            inter = s.rect & o.rect
            if not inter.is_empty and inter.get_area() > 0.6 * max(min(s.rect.get_area(), o.rect.get_area()), 0.01):
                dup = True
                break
        if not dup:
            by_text.setdefault((s.text, s.ocr), []).append(s)
            keep.append(s)
    return keep


def needs_space(prev, cur):
    if prev.trail or cur.lead:
        return True
    return cur.x0 - prev.x1 > 0.12 * min(prev.size, cur.size)


def join_plain(spans):
    out = ""
    for i, s in enumerate(spans):
        if i and needs_space(spans[i - 1], s):
            out += " "
        out += s.text
    return re.sub(r"\u0338\s*=", "≠", out).replace("\u0338", "/")


def norm_key(text):
    return re.sub(r"\s+", " ", re.sub(r"\d+", "#", text.lower())).strip()


# ---------------------------------------------------------------- math

def is_core_math(s):
    if s.ocr:
        return False
    if MATH_CORE_RE.search(s.font):
        return True
    return any(ch in MATH_OP_CHARS for ch in s.text)


def is_math_compatible(s, base_size):
    if is_core_math(s):
        return True
    if s.size < 0.8 * base_size:
        return True
    letters = re.findall(r"[A-Za-z]+", s.text)
    if not letters:
        return True
    if MATH_COMPAT_RE.search(s.font):
        return all(w in FUNC_NAMES or len(w) == 1 for w in letters)
    return False


def find_math_runs(spans):
    """Returns a list of (start, end) index ranges of math runs inside a span list."""
    if not spans:
        return []
    base = max(s.size for s in spans)
    core = [is_core_math(s) for s in spans]
    compat = [is_math_compatible(s, base) for s in spans]
    in_run = [False] * len(spans)
    for i, c in enumerate(core):
        if not c:
            continue
        in_run[i] = True
        j = i - 1
        while j >= 0 and compat[j] and not in_run[j]:
            in_run[j] = True
            j -= 1
        j = i + 1
        while j < len(spans) and compat[j] and not core[j]:
            in_run[j] = True
            j += 1
    runs, i = [], 0
    while i < len(spans):
        if in_run[i]:
            j = i
            while j + 1 < len(spans) and in_run[j + 1]:
                j += 1
            text = "".join(s.text for s in spans[i:j + 1])
            if any(ch.isalnum() for ch in text) or len(text.strip()) > 2:
                runs.append((i, j + 1))
            i = j + 1
        else:
            i += 1
    return runs


def span_latex(s):
    font = s.font
    bb = bool(re.match(r"MSBM", font, re.I))
    cal = bool(re.match(r"CMSY|CMBSY", font, re.I))
    frak = bool(re.match(r"EUFM", font, re.I))
    bold = s.bold
    text = s.text
    for fn in sorted(FUNC_NAMES, key=len, reverse=True):
        text = re.sub(rf"(?<![A-Za-z\\]){fn}(?![A-Za-z])", "\x00" + fn + "\x01", text)
    out, i = [], 0
    while i < len(text):
        ch = text[i]
        if ch == "\x00":
            j = text.index("\x01", i)
            out.append("\\" + text[i + 1:j] + " ")
            i = j + 1
            continue
        if ch in GREEK:
            tok = "\\" + GREEK[ch] + " "
            out.append(r"\boldsymbol{" + tok.strip() + "}" if bold else tok)
        elif ch in LATEX_CHARS:
            out.append(LATEX_CHARS[ch] + (" " if LATEX_CHARS[ch][-1:].isalpha() else ""))
        elif ch.isalpha() and ch.isascii():
            if bb:
                out.append(r"\mathbb{" + ch + "}")
            elif cal and ch.isupper():
                out.append(r"\mathcal{" + ch + "}")
            elif frak:
                out.append(r"\mathfrak{" + ch + "}")
            elif bold:
                out.append(r"\mathbf{" + ch + "}")
            else:
                out.append(ch)
        elif ch.isspace():
            out.append(" ")
        else:
            out.append(ch)
        i += 1
    return "".join(out)


def rule_latex(spans):
    spans = sorted(spans, key=lambda s: (round(s.x0), -s.oy))
    base = max(s.size for s in spans)
    mains = [s for s in spans if s.size >= 0.8 * base]
    base_y = sorted(s.oy for s in mains)[len(mains) // 2]
    out, script_kind, script_buf = [], None, []

    def flush():
        nonlocal script_kind, script_buf
        if script_kind:
            body = "".join(script_buf).strip()
            out.append(script_kind + ("{" + body + "}" if len(body) != 1 else body))
        script_kind, script_buf = None, []

    prev = None
    for s in spans:
        kind = None
        if s.size < 0.8 * base:
            dy = s.oy - base_y
            kind = "^" if dy < -0.12 * base else ("_" if dy > 0.08 * base else None)
        tok = span_latex(s)
        if kind:
            if kind != script_kind:
                flush()
                script_kind = kind
            elif prev is not None and needs_space(prev, s):
                script_buf.append(" ")
            script_buf.append(tok)
        else:
            flush()
            if prev is not None and needs_space(prev, s):
                out.append(" ")
            out.append(tok)
        prev = s
    flush()
    tex = "".join(out)
    tex = re.sub(r"\.\s*\.\s*\.", r"\\ldots ", tex)
    tex = re.sub(r"[ \t]+", " ", tex).strip()
    tex = re.sub(r" ?([_^]) ?", r"\1", tex)
    return re.sub(r"\\not ?=", r"\\neq ", tex).strip()


def latex_signature(tex):
    t = re.sub(r"\\(mathbf|mathbb|mathcal|mathrm|mathit|boldsymbol|bm|operatorname|text|mathfrak|left|right|"
               r"big|Big|bigg|Bigg|displaystyle|mathsf)\b", "", tex)
    t = re.sub(r"\\[a-zA-Z]+", "", t)
    return Counter(ch for ch in t if ch.isalnum())


def signature_similarity(a, b):
    sa, sb = latex_signature(a), latex_signature(b)
    total = sum(sa.values()) + sum(sb.values())
    if not total:
        return 1.0
    return 2 * sum((sa & sb).values()) / total


def braces_balanced(tex):
    depth = 0
    for i, ch in enumerate(tex):
        if ch == "{" and (i == 0 or tex[i - 1] != "\\"):
            depth += 1
        elif ch == "}" and (i == 0 or tex[i - 1] != "\\"):
            depth -= 1
            if depth < 0:
                return False
    return depth == 0


def multirow_latex(spans):
    rows = build_rows(spans)
    return r" \\ ".join(rule_latex(r) for r in rows) if len(rows) > 1 else rule_latex(spans)


def is_complex_math(spans, bars):
    """True when the formula has 2-D structure that a left-to-right reading cannot rebuild."""
    if len(build_rows(spans)) > 1:
        return True
    if any(re.match(r"CMEX|EUEX", s.font, re.I) or any(ch in "√∑∏∫" for ch in s.text) for s in spans):
        return True
    r = pymupdf.Rect(min(s.x0 for s in spans), min(s.y0 for s in spans),
                     max(s.x1 for s in spans), max(s.y1 for s in spans)) + (-1, -1, 1, 1)
    return any(b in r for b in bars)


class MathEngine:
    """mode: "auto" uses pix2tex only for 2-D formulas, "pix2tex" tries it on every formula, "rules" never."""

    def __init__(self, latex_ocr=None, mode="auto"):
        self.latex_ocr = latex_ocr
        self.mode = mode if latex_ocr is not None else "rules"
        self._bars = {}

    def bars(self, page):
        if page.number not in self._bars:
            self._bars[page.number] = [r for r in (pymupdf.Rect(d["rect"]) for d in page.get_drawings())
                                       if r.height < 1.2 and r.width > 3]
        return self._bars[page.number]

    def worth_pix2tex(self, spans, display, complex_):
        if self.mode == "rules":
            return False
        if self.mode == "auto":
            return complex_
        base = max(s.size for s in spans)
        has_script = any(s.size < 0.8 * base for s in spans)
        return display or complex_ or has_script or sum(len(s.text) for s in spans) >= 6

    def convert(self, page, spans, stats, display=False):
        complex_ = is_complex_math(spans, self.bars(page))
        rule = multirow_latex(spans)
        if complex_:
            stats.complex_formulas += 1
        if not self.worth_pix2tex(spans, display, complex_):
            return rule
        min_similarity = 0.75 if complex_ else 0.9
        base = max(s.size for s in spans)
        r = pymupdf.Rect(min(s.x0 for s in spans), min(s.y0 for s in spans),
                         max(s.x1 for s in spans), max(s.y1 for s in spans))
        r = (r + (-0.25 * base, -0.15 * base, 0.25 * base, 0.15 * base)) & page.rect
        try:
            from PIL import Image
            zoom = max(2.0, 30.0 / base)
            pix = page.get_pixmap(matrix=pymupdf.Matrix(zoom, zoom), clip=r, alpha=False)
            img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
            tex = (self.latex_ocr(img) or "").strip()
        except Exception as e:
            stats.warnings.append(f"pix2tex failed on page {page.number + 1}: {e}")
            stats.pix2tex_rejected += 1
            return rule
        tex = re.sub(r"[\s.,;:]+$", "", tex)
        sim = signature_similarity(tex, rule)
        ok = bool(tex) and braces_balanced(tex) and len(tex) <= 4 * len(rule) + 40 and sim >= min_similarity
        stats.math_log.append({"page": page.number + 1, "rules": rule, "pix2tex": tex, "similarity": round(sim, 2),
                               "accepted": ok})
        if ok:
            stats.pix2tex_used += 1
            return tex
        stats.pix2tex_rejected += 1
        return rule


# ---------------------------------------------------------------- inline rendering

def render_spans(page, spans, engine, stats, allow_bold=True, display_ok=False):
    """Renders a span list as Markdown text with $...$ math. Returns (text, is_pure_math, latex)."""
    runs = find_math_runs(spans)
    pieces = []  # (first_span, last_span, text, kind)
    i, ri = 0, 0
    run_starts = {a: b for a, b in runs}
    while i < len(spans):
        if i in run_starts:
            j = run_starts[i]
            pieces.append((spans[i], spans[j - 1], spans[i:j], "math"))
            i = j
        else:
            pieces.append((spans[i], spans[i], [spans[i]], "bold" if spans[i].bold else "text"))
            i += 1
    pure = len(pieces) == 1 and pieces[0][3] == "math"
    if pure and display_ok:
        tex = engine.convert(page, pieces[0][2], stats, display=True)
        tex, tail = peel_punct(tex)
        stats.formulas_display += 1
        return tail, True, tex
    out = ""
    prev_last = None
    merged = []
    for first, last, sp, kind in pieces:
        if merged and merged[-1][3] == kind and kind != "math":
            merged[-1] = (merged[-1][0], last, merged[-1][2] + sp, kind)
        else:
            merged.append((first, last, sp, kind))
    for first, last, sp, kind in merged:
        if prev_last is not None and needs_space(prev_last, first):
            out += " "
        if kind == "math":
            tex = engine.convert(page, sp, stats, display=False)
            tex, tail = peel_punct(tex)
            stats.formulas_inline += 1
            out += f"${tex}$" + tail if tex else tail
        elif kind == "bold" and allow_bold:
            out += f"**{join_plain(sp)}**"
        else:
            out += join_plain(sp)
        prev_last = last
    return out, False, None


def peel_punct(tex):
    m = re.search(r"[.,;:]+$", tex)
    if m and not tex.endswith(r"\ldots"):
        return tex[:m.start()].rstrip(), m.group(0)
    return tex, ""


# ---------------------------------------------------------------- figures

def rect_key(r, nd=0):
    return tuple(round(v, nd) for v in (r.x0, r.y0, r.x1, r.y1))


def page_graphics(page):
    """Yields (key, rect) for every image placement and visible drawing on the page."""
    area = page.rect.width * page.rect.height
    items = []
    try:
        infos = page.get_image_info(hashes=True, xrefs=True)
    except Exception:
        infos = page.get_image_info()
    for info in infos:
        r = pymupdf.Rect(info["bbox"]) & page.rect
        if r.is_empty:
            continue
        digest = info.get("digest")
        items.append((("img", digest.hex() if isinstance(digest, (bytes, bytearray)) else info.get("xref"),
                       rect_key(r)), r))
    for d in page.get_drawings():
        r = pymupdf.Rect(d["rect"]) & page.rect
        if r.width * r.height > 0.5 * area:
            continue
        fill, color = d.get("fill"), d.get("color")
        if color is None and (fill is None or tuple(fill) == (1.0, 1.0, 1.0)):
            continue
        if r.width < 0.5:
            r.x1 = r.x0 + 0.5
        if r.height < 0.5:
            r.y1 = r.y0 + 0.5
        items.append((("drw", rect_key(r)), r))
    return items


def cluster_rects(rects, margin):
    clusters = []
    for r in rects:
        r = pymupdf.Rect(r)
        grown = r + (-margin, -margin, margin, margin)
        hit = [c for c in clusters if c.intersects(grown)]
        for c in hit:
            r |= c
            clusters.remove(c)
        clusters.append(r)
    changed = True
    while changed:
        changed = False
        for i in range(len(clusters)):
            for j in range(i + 1, len(clusters)):
                a, b = clusters[i], clusters[j]
                if (a + (-margin, -margin, margin, margin)).intersects(b):
                    clusters[i] = a | b
                    del clusters[j]
                    changed = True
                    break
            if changed:
                break
    return clusters


def center_in(s, r, pad=2):
    return r.x0 - pad <= s.cx <= r.x1 + pad and r.y0 - pad <= s.cy <= r.y1 + pad


def rect_distance(a, b):
    dx = max(b.x0 - a.x1, a.x0 - b.x1, 0)
    dy = max(b.y0 - a.y1, a.y0 - b.y1, 0)
    return max(dx, dy)


def visible_bbox(page, rect, bg=None):
    """Shrinks a rect to its visibly drawn content (image placements are often clipped)."""
    from PIL import Image, ImageChops
    if rect.width < 2 or rect.height < 2:
        return rect
    zoom = 1.0 if rect.width * rect.height > 4000 else 3.0
    pix = page.get_pixmap(matrix=pymupdf.Matrix(zoom, zoom), clip=rect, alpha=False)
    img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
    if bg is None:
        bg = page_background(page)
    diff = ImageChops.difference(img, Image.new("RGB", img.size, bg)).convert("L").point(lambda p: 255 if p > 24 else 0)
    box = diff.getbbox()
    if not box:
        return pymupdf.Rect()
    sx, sy = rect.width / pix.width, rect.height / pix.height
    return pymupdf.Rect(rect.x0 + box[0] * sx, rect.y0 + box[1] * sy, rect.x0 + box[2] * sx, rect.y0 + box[3] * sy)


def text_free_copy(page):
    """Returns (doc, page): a one-page copy with all text removed, used to measure graphics only."""
    tmp = pymupdf.open()
    tmp.insert_pdf(page.parent, from_page=page.number, to_page=page.number)
    p = tmp[0]
    try:
        p.add_redact_annot(p.rect)
        p.apply_redactions(images=pymupdf.PDF_REDACT_IMAGE_NONE, graphics=pymupdf.PDF_REDACT_LINE_ART_NONE,
                           text=pymupdf.PDF_REDACT_TEXT_REMOVE)
    except Exception:
        pass
    return tmp, p


def page_background(page):
    pix = page.get_pixmap(matrix=pymupdf.Matrix(0.2, 0.2), alpha=False)
    corners = [pix.pixel(0, 0), pix.pixel(pix.width - 1, 0), pix.pixel(0, pix.height - 1),
               pix.pixel(pix.width - 1, pix.height - 1)]
    return Counter(corners).most_common(1)[0][0]


# ---------------------------------------------------------------- bullets, headings, columns

def detect_bullet(seg, markers):
    first = seg.spans[0]
    t = first.text
    if len(t) == 1 and (t in BULLET_GLYPHS or t == "\ufffd" or "\uf000" <= t <= "\uf0ff" or
                        (t in DASH_BULLETS and len(seg.spans) > 1)):
        if len(seg.spans) > 1:
            seg.bullet, seg.bullet_x = "glyph", first.x0
            return
    if re.match(r"^([•●○◦▪▫■□►▸▹‣⁃∙·➢➤❖✓✔◆◇\uf000-\uf0ff]|[-–—*])\s+\S", t):
        m = re.match(r"^\S+\s+", t)
        first.text = t[m.end():]
        first.lead = False
        seg.bullet, seg.bullet_x = "inline", first.x0
        return
    if NUMBERED_RE.match(seg.plain):
        seg.bullet, seg.bullet_x = "number", first.x0
        return
    for m in markers:
        if m.x1 <= seg.x0 + 1 and seg.x0 - m.x1 <= 12 and m.y1 > seg.y0 and m.y0 < seg.y1:
            seg.bullet, seg.bullet_x = "marker", m.x0
            return


def find_gutter(segs, width):
    if len(segs) < 4:
        return None
    best = None
    for x in range(int(width * 0.3), int(width * 0.7), 2):
        if any(s.x0 < x < s.x1 for s in segs):
            continue
        left = [s for s in segs if s.x1 <= x]
        right = [s for s in segs if s.x0 >= x]
        if len(left) < 2 or len(right) < 2:
            continue
        ly0, ly1 = min(s.y0 for s in left), max(s.y1 for s in left)
        ry0, ry1 = min(s.y0 for s in right), max(s.y1 for s in right)
        ov = min(ly1, ry1) - max(ly0, ry0)
        if ov < 0.5 * min(ly1 - ly0, ry1 - ry0):
            continue
        score = min(len(left), len(right))
        if best is None or score > best[0]:
            best = (score, x)
    return best[1] if best else None


def cluster_levels(xs, tol=3.0):
    levels = []
    for x in sorted(xs):
        if not levels or x - levels[-1][-1] > tol:
            levels.append([x])
        else:
            levels[-1].append(x)
    return [sum(l) / len(l) for l in levels]


# ---------------------------------------------------------------- page conversion

@dataclass
class SlideResult:
    number: int
    title: str
    body: list
    figures: list
    signature: list
    links: list


def cell_text(page, spans, engine, stats):
    parts = []
    for seg in sorted(make_segments(spans), key=lambda s: (s.row, s.x0)):
        parts.append(render_spans(page, seg.spans, engine, stats, allow_bold=False)[0].strip())
    return " ".join(p for p in parts if p)


def grid_to_md(grid):
    width = max(len(r) for r in grid)
    rows = [[c.replace("|", r"\|").replace("\n", " ") for c in r] + [""] * (width - len(r)) for r in grid]
    lines = ["| " + " | ".join(rows[0]) + " |", "|" + "---|" * width]
    lines += ["| " + " | ".join(r) + " |" for r in rows[1:]]
    return "\n".join(lines)


def find_borderless_tables(col_segs):
    """Finds runs of rows whose segments line up in >= 2 columns. Returns list of (row_ids, column_starts)."""
    rows = {}
    for s in col_segs:
        rows.setdefault(s.row, []).append(s)
    order = sorted(rows, key=lambda r: min(s.y0 for s in rows[r]))
    tables, i = [], 0
    while i < len(order):
        segs = sorted(rows[order[i]], key=lambda s: s.x0)
        if len(segs) < 3 or any(s.bullet for s in segs):
            i += 1
            continue
        cols = [s.x0 for s in segs]
        size = max(s.size for s in segs)
        members, last_y1, j = [order[i]], max(s.y1 for s in segs), i + 1
        while j < len(order):
            nxt = sorted(rows[order[j]], key=lambda s: s.x0)
            y0 = min(s.y0 for s in nxt)
            if y0 - last_y1 > 2.5 * size or any(s.bullet for s in nxt):
                break
            if not all(any(abs(s.x0 - c) <= 4 for c in cols) for s in nxt):
                break
            members.append(order[j])
            last_y1 = max(s.y1 for s in nxt)
            j += 1
        full = sum(len(rows[r]) >= 2 for r in members)
        if full >= 3 and (len(cols) >= 3 or full >= 4):
            if i > 0 and not (tables and order[i - 1] in tables[-1][0]):
                above = sorted(rows[order[i - 1]], key=lambda s: s.x0)
                gap = min(s.y0 for s in segs) - max(s.y1 for s in above)
                if (len(above) >= 2 and gap <= 2.5 * size and not any(s.bullet for s in above)
                        and all(any(c - 4 <= s.x0 and s.x0 < c2 for c, c2 in zip(cols, cols[1:] + [1e9])) for s in above)):
                    members.insert(0, order[i - 1])
            tables.append((members, cols))
            i = j
        else:
            i += 1
    return tables, rows


def is_pure_math(seg):
    cs = seg.content_spans
    runs = find_math_runs(cs)
    return bool(cs) and len(runs) == 1 and runs[0] == (0, len(cs))


def merge_stacked_math(segs):
    """Joins formula pieces laid out in 2-D (fractions, limits, radicals) into the line they belong to."""
    segs = sorted(segs, key=lambda s: (s.y0, s.x0))
    changed = True
    while changed:
        changed = False
        for s in segs:
            if s.bullet or not is_pure_math(s):
                continue
            for o in segs:
                if o is s:
                    continue
                cs = o.content_spans
                runs = find_math_runs(cs)
                if not runs:
                    continue
                math = [sp for a, b in runs for sp in cs[a:b]]
                mx0, mx1 = min(sp.x0 for sp in math), max(sp.x1 for sp in math)
                my0, my1 = min(sp.y0 for sp in math), max(sp.y1 for sp in math)
                size = max(o.size, s.size)
                h_overlap = min(mx1, s.x1) - max(mx0, s.x0)
                v_gap = max(s.y0 - my1, my0 - s.y1)
                v_overlap = min(my1, s.y1) - max(my0, s.y0)
                h_gap = max(s.x0 - mx1, mx0 - s.x1)
                stacked = h_overlap > 0.3 * min(mx1 - mx0, s.x1 - s.x0) and v_gap < 0.35 * size
                beside = v_overlap >= 0.5 * min(my1 - my0, s.y1 - s.y0) and h_gap < size
                if stacked or beside:
                    keep, drop = (o, s) if len(o.spans) >= len(s.spans) or not is_pure_math(o) else (s, o)
                    keep.spans = sorted(keep.spans + drop.spans, key=lambda sp: (sp.x0, sp.y0))
                    segs.remove(drop)
                    changed = True
                    break
            if changed:
                break
    return segs


def ocr_line_ok(text):
    toks = text.split()
    if len(toks) < 3:
        return False
    good = sum(bool(re.fullmatch(r"[(\[\"'“]?([A-Za-z][a-z'’-]*|[A-Z]{2,}|\d+([.,]\d+)?%?|[A-Za-z]\.|[=+\-–<>])"
                                  r"[)\]\"'”.,;:?!]*", t)) for t in toks)
    return good / len(toks) >= 0.75


def convert_page(page, ctx, engine, stats, opts):
    W, H = page.rect.width, page.rect.height
    spans = extract_spans(page)
    native = [s for s in spans if not s.ocr]
    ocr = [s for s in spans if s.ocr]
    native = [s for s in native if id(s) not in ctx["furniture"].get(page.number, set())]
    native = [s for s in native if not any(s.rect in pymupdf.Rect(r) for r in ctx["furniture_rects"].get(page.number, []))]

    tables, table_rects = [], []
    if opts.get("detect_tables", True):
        try:
            for t in page.find_tables().tables:
                grid = []
                for row in t.rows:
                    cells = []
                    for c in row.cells:
                        inside = [s for s in native if c is not None and center_in(s, pymupdf.Rect(c), pad=0)]
                        cells.append(cell_text(page, inside, engine, stats))
                    grid.append(cells)
                flat = [c for r in grid for c in r]
                if len(grid) >= 2 and max(len(r) for r in grid) >= 2 and sum(bool(c) for c in flat) >= 0.5 * len(flat):
                    tr = pymupdf.Rect(t.bbox)
                    table_rects.append(tr)
                    tables.append((tr.y0, grid_to_md(grid)))
        except Exception as e:
            stats.warnings.append(f"table detection failed on page {page.number + 1}: {e}")
    native = [s for s in native if not any(center_in(s, tr) for tr in table_rects)]
    stats.tables += len(tables)

    body_size = ctx["body_size"]
    rotated = [s for s in native if s.rotated]
    segs = make_segments([s for s in native if not s.rotated])

    title_segs = []
    top = [s for s in segs if s.y1 <= H * 0.35]
    if top:
        mx = max(s.size for s in top)
        if mx >= 1.15 * body_size:
            first = min((s for s in top if s.size >= 0.95 * mx), key=lambda s: s.y0)
            title_segs = [first]
            for s in sorted(top, key=lambda s: s.y0):
                if s is not first and s.size >= 0.95 * mx and 0 <= s.y0 - title_segs[-1].y1 < 0.6 * mx:
                    title_segs.append(s)
    segs = [s for s in segs if s not in title_segs]
    title = " ".join(render_spans(page, s.spans, engine, stats, allow_bold=False)[0] for s in title_segs)
    title_bottom = max((s.y1 for s in title_segs), default=0)

    tmp_doc, bare = text_free_copy(page)
    bg = page_background(bare)
    gfx, rasters, drawings = [], [], []
    for key, r in page_graphics(page):
        if key in ctx["template_graphics"] or any(r in tr for tr in table_rects):
            continue
        if key[0] == "img":
            r = visible_bbox(bare, r, bg)
            if r.is_empty:
                continue
            rasters.append(r)
        else:
            drawings.append(r)
        gfx.append(r)
    tmp_doc.close()
    thin = [r for r in drawings if min(r.width, r.height) < 1.5 and (r.width > 0.4 * W or r.height > 0.4 * H)]
    clusters = cluster_rects([r for r in gfx if not any(r is t for t in thin)], margin=12)
    for t in thin:
        for k, c in enumerate(clusters):
            if (c + (-3, -3, 3, 3)).intersects(t):
                clusters[k] = c | t
                break
    if title_bottom:
        clusters = [pymupdf.Rect(c.x0, max(c.y0, title_bottom + 1), c.x1, c.y1) if c.y1 > title_bottom + 5 else c
                    for c in clusters]
    min_area = opts.get("min_figure_area", 0.01) * W * H
    figs = [c for c in clusters if c.width * c.height >= min_area and min(c.width, c.height) >= 12]
    markers = [c for c in clusters if c not in figs and c.width <= 12 and c.height <= 12]

    def is_text_box(f):
        if any(f.intersects(r) for r in rasters):
            return False
        frames = [r for r in drawings if r in f and r.get_area() >= 0.6 * f.get_area()]
        inside = [s for s in segs if any(center_in(s, r, pad=0) for r in frames)]
        n_draw = sum(1 for r in drawings if r.intersects(f))
        return bool(frames) and n_draw <= 8 and sum(len(s.plain.split()) for s in inside) >= 8

    figs = [f for f in figs if not is_text_box(f)]

    label_spans = {i: [] for i in range(len(figs))}
    for s in rotated:
        near = [(rect_distance(s.rect, f), i) for i, f in enumerate(figs)]
        if near and min(near)[0] < 30:
            label_spans[min(near)[1]].append(s)

    def label_like(seg):
        return len(seg.plain.split()) <= 6 and not re.match(r"^[•●▪◦\-–]\s", seg.plain)

    for seg in list(segs):
        for i, f in enumerate(figs):
            on_raster = (any(center_in(seg, r, pad=0) for r in rasters) and len(seg.plain.split()) <= 12
                         and not re.match(r"^[•●▪◦\-–]\s", seg.plain))
            if center_in(seg, f) and (label_like(seg) or on_raster):
                label_spans[i].extend(seg.spans)
                figs[i] = f | seg.rect
                segs.remove(seg)
                break
    for _ in range(3):
        moved = False
        for seg in list(segs):
            if len(seg.plain.split()) > 4 or not label_like(seg):
                continue
            for i, f in enumerate(figs):
                if rect_distance(seg.rect, f) <= 10:
                    label_spans[i].extend(seg.spans)
                    figs[i] = f | seg.rect
                    segs.remove(seg)
                    moved = True
                    break
        if not moved:
            break
    merged = True
    while merged:
        merged = False
        for i in range(len(figs)):
            for j in range(i + 1, len(figs)):
                if figs[i].intersects(figs[j]):
                    figs[i] |= figs[j]
                    label_spans[i].extend(label_spans.pop(j))
                    label_spans = {k if k < j else k - 1: v for k, v in sorted(label_spans.items())}
                    del figs[j]
                    merged = True
                    break
            if merged:
                break
    figs = [f & page.rect for f in figs]

    for s in segs:
        detect_bullet(s, markers)
    weights = Counter()
    for seg in segs:
        for s in seg.spans:
            weights[round(s.size * 2) / 2] += len(s.text)
    page_body = max(weights.items(), key=lambda kv: kv[1])[0] if weights else body_size
    gutter = find_gutter(segs, W)
    columns = [segs] if gutter is None else [[s for s in segs if s.x1 <= gutter], [s for s in segs if s.x1 > gutter]]

    blocks = []
    for col in columns:
        units = []
        found, rows = find_borderless_tables(col)
        in_table = set()
        for members, cols in found:
            grid = []
            for r in members:
                cells = [[] for _ in cols]
                for seg in rows[r]:
                    for s in seg.spans:
                        k = max([c for c in range(len(cols)) if cols[c] - 4 <= s.x0], default=0)
                        cells[k].append(s)
                texts = [cell_text(page, c, engine, stats) for c in cells]
                if grid and not texts[0] and any(texts):
                    grid[-1] = [(a + " " + b).strip() for a, b in zip(grid[-1], texts)]
                else:
                    grid.append(texts)
            in_table.update(members)
            units.append((min(s.y0 for r in members for s in rows[r]), "table", grid_to_md(grid)))
            stats.tables += 1
        rest = []
        for r, row_segs in rows.items():
            if r in in_table:
                continue
            row_segs.sort(key=lambda s: s.x0)
            cur = row_segs[0]
            for s in row_segs[1:]:
                if not s.bullet and s.x0 - cur.x1 < 3.5 * max(min(s.size, cur.size), 4):
                    cur = Segment(cur.spans + s.spans, cur.bullet, cur.bullet_x, cur.row)
                else:
                    rest.append(cur)
                    cur = s
            rest.append(cur)
        rest = merge_stacked_math(rest)
        units += [(s.y0, "seg", s) for s in rest]
        units.sort(key=lambda u: (u[0], u[2].x0 if u[1] == "seg" else 0))
        levels = cluster_levels([s.bullet_x for s in rest if s.bullet])

        if blocks and blocks[-1]["kind"] == "para" and not re.search(r"[.!?:]$", blocks[-1]["text"]):
            last = blocks[-1]
            last["text_x"] = min(s.x0 for s in col) if col else last["text_x"]
            last["y1"] = min((s.y0 for s in col), default=last["y1"])
        else:
            last = None
        for y0, kind, seg in units:
            if kind == "table":
                last = {"kind": "table", "text": seg, "y0": y0}
                blocks.append(last)
                continue
            content = seg.content_spans
            if not content:
                continue
            size = seg.size
            if seg.bullet:
                lvl = min(range(len(levels)), key=lambda k: abs(levels[k] - seg.bullet_x))
                text, _, _ = render_spans(page, content, engine, stats)
                last = {"kind": "item", "level": lvl, "text": text, "text_x": seg.text_x,
                        "y0": seg.y0, "y1": seg.y1, "size": size}
                blocks.append(last)
                continue
            cont = (last is not None and last["kind"] in ("item", "para")
                    and seg.y0 - last["y1"] <= 0.9 * size and abs(size - last["size"]) <= 1.5
                    and (abs(seg.x0 - last["text_x"]) <= 3 or
                         (last["kind"] == "para" and abs(seg.x0 - last["text_x"]) <= 8)))
            first = content[0]
            plain = seg.plain.strip()
            avg = sum(s.size * len(s.text) for s in content) / max(sum(len(s.text) for s in content), 1)
            is_heading = ((first.size >= 1.2 * page_body and first.bold) or avg >= 1.5 * page_body
                          or (all(s.bold for s in content) and avg >= 1.1 * page_body and len(plain.split()) <= 8))
            is_heading = (is_heading and not cont and len(plain.split()) <= 12 and not find_math_runs(content)
                          and not plain[:1].islower() and not re.search(r"[.,;]$", plain))
            if is_heading:
                text, _, _ = render_spans(page, content, engine, stats, allow_bold=False)
                last = {"kind": "heading", "text": text, "y0": seg.y0, "y1": seg.y1, "size": size}
                blocks.append(last)
                continue
            text, pure, tex = render_spans(page, content, engine, stats, display_ok=not cont)
            if pure:
                last = {"kind": "math", "text": f"$${tex}$${text}", "y0": seg.y0, "y1": seg.y1, "size": size}
                blocks.append(last)
                continue
            if cont:
                sep = "" if re.search(r"[A-Za-z]-$", last["text"]) else " "
                last["text"] += sep + text
                last["y1"] = seg.y1
            else:
                last = {"kind": "para", "text": text, "text_x": seg.x0, "y0": seg.y0, "y1": seg.y1, "size": size}
                blocks.append(last)

    for y0, md in tables:
        idx = next((k for k, b in enumerate(blocks) if b["y0"] > y0), len(blocks))
        blocks.insert(idx, {"kind": "table", "text": md, "y0": y0})

    links = []
    for ln in page.get_links():
        uri = ln.get("uri")
        if not uri:
            continue
        r = pymupdf.Rect(ln["from"])
        txt = join_plain(sorted([s for s in native if r.intersects(s.rect)], key=lambda s: (s.y0, s.x0))).strip()
        links.append((txt or uri, uri))

    figures = []
    for i, f in enumerate(sorted(range(len(figs)), key=lambda k: (figs[k].y0, figs[k].x0))):
        rect = figs[f]
        labels = []
        if opts.get("include_figure_labels", True):
            for seg in make_segments([s for s in label_spans[f] if not s.rotated]):
                t, _, _ = render_spans(page, seg.spans, engine, stats, allow_bold=False)
                t = t.strip()
                if t and t not in labels:
                    labels.append(t)
        ocr_lines = []
        if opts.get("use_ocr_layer", True):
            cand = [s for s in ocr if center_in(s, rect, pad=0) and any(center_in(s, r, pad=0) for r in rasters)
                    and not any(s.rect.intersects(n.rect) and (s.rect & n.rect).get_area() > 0.3 * max(s.rect.get_area(), 0.1)
                                for n in native)]
            for seg in sorted(make_segments(cand), key=lambda s: (s.y0, s.x0)):
                t = seg.plain.strip()
                if ocr_line_ok(t):
                    ocr_lines.append(t)
            if len(ocr_lines) < 3 or sum(len(l.split()) for l in ocr_lines) < 20:
                ocr_lines = []
            else:
                stats.ocr_image_texts += 1
        figures.append({"rect": rect, "labels": labels, "ocr": ocr_lines})

    signature = [norm_key(b["text"]) for b in blocks]
    return SlideResult(page.number + 1, title.strip(), blocks, figures, signature, links)


# ---------------------------------------------------------------- document level

def analyse_document(doc):
    n = len(doc)
    furniture, furniture_rects, page_labels = {}, {}, {}
    seg_keys, graphics_count = Counter(), Counter()
    per_page = []
    sizes = Counter()
    for page in doc:
        H = page.rect.height
        native = [s for s in extract_spans(page) if not s.ocr and not s.rotated]
        segs = make_segments(native)
        band = [s for s in segs if s.rect.y0 + s.rect.height / 2 < 0.1 * H or s.rect.y0 + s.rect.height / 2 > 0.9 * H]
        keys = {(norm_key(s.plain), "top" if s.y0 < H / 2 else "bottom"): s for s in band}
        per_page.append((native, keys))
        seg_keys.update(keys.keys())
        graphics_count.update({k for k, _ in page_graphics(page)})
        for s in native:
            sizes[round(s.size * 2) / 2] += len(s.text)
    threshold = max(2, 0.5 * n)
    template_graphics = {k for k, c in graphics_count.items() if c >= threshold} if n >= 3 else set()
    for pno, (native, keys) in enumerate(per_page):
        ids, rects = set(), []
        if n >= 3:
            for key, seg in keys.items():
                if seg_keys[key] >= threshold:
                    ids.update(id(s) for s in seg.spans)
                    rects.append(seg.rect + (-0.5, -0.5, 0.5, 0.5))
                    if key[0] == "#":
                        page_labels[pno] = seg.plain.strip()
        furniture[pno] = ids
        furniture_rects[pno] = rects
    for pno, (native, keys) in enumerate(per_page):
        for s in native:
            if id(s) in furniture[pno]:
                sizes[round(s.size * 2) / 2] -= len(s.text)
    body_size = max(sizes.items(), key=lambda kv: kv[1])[0] if sizes else 10.0
    return {"furniture": {}, "furniture_rects": furniture_rects, "template_graphics": template_graphics,
            "body_size": body_size, "page_labels": page_labels}


def iou(a, b):
    inter = a & b
    if inter.is_empty:
        return 0.0
    union = a.get_area() + b.get_area() - inter.get_area()
    return inter.get_area() / union if union else 0.0


def safe_name(stem):
    return re.sub(r'[\\/:*?"<>|#^\[\]\s]+', "_", stem).strip("_") or "document"


def blocks_to_md(blocks):
    lines, prev, prev_level = [], None, -1
    for b in blocks:
        if b["kind"] == "item":
            if prev is not None and prev != "item":
                lines.append("")
                prev_level = -1
            level = min(b["level"], prev_level + 1)
            prev_level = level
            lines.append("  " * level + "- " + b["text"].strip())
        else:
            if prev is not None:
                lines.append("")
            if b["kind"] == "heading":
                lines.append("### " + b["text"].strip())
            else:
                lines.append(re.sub(r"^#", r"\\#", b["text"].strip()))
        prev = b["kind"]
    return re.sub(r"\*\* \*\*", " ", "\n".join(lines))


def shared_output_name(stems, requested=""):
    requested = (requested or "").strip()
    if requested.lower().endswith(".md"):
        requested = requested[:-3].strip()
    if requested:
        return safe_name(requested)
    stems = [safe_name(s) for s in stems]
    if not stems:
        return "slides"
    if len(stems) == 1:
        return stems[0]
    prefix = stems[0]
    for stem in stems[1:]:
        while prefix and not stem.startswith(prefix):
            prefix = prefix[:-1]
    prefix = prefix.rstrip("-_ ")
    if len(prefix) < 3:
        return "slides"
    return safe_name(prefix)


def convert_pdf(pdf_path, out_dir, opts, latex_ocr=None):
    pdf_path, out_dir = Path(pdf_path), Path(out_dir)
    name = safe_name(pdf_path.stem)
    img_dir = out_dir / "images"
    img_dir.mkdir(parents=True, exist_ok=True)
    stats = Stats()
    engine = MathEngine(latex_ocr if opts.get("use_pix2tex", True) else None, mode=opts.get("math_engine", "auto"))
    doc = pymupdf.open(pdf_path)
    ctx = analyse_document(doc)

    slides = []
    for page in doc:
        if not any(not s.ocr for s in extract_spans(page)):
            stats.warnings.append(f"page {page.number + 1} has no native text (scanned?)")
        try:
            slides.append((page, convert_page(page, ctx, engine, stats, opts)))
        except Exception as e:
            stats.warnings.append(f"page {page.number + 1} failed ({type(e).__name__}: {e}); plain text used")
            text = "\n".join(l.strip() for l in page.get_text("text", sort=True).splitlines() if l.strip())
            body = [{"kind": "para", "text": text, "y0": 0}]
            slides.append((page, SlideResult(page.number + 1, "", body, [], [norm_key(text)], [])))

    dropped = []
    if opts.get("drop_build_slides", True):
        keep = []
        for k, (page, sr) in enumerate(slides):
            if k + 1 < len(slides):
                nxt = slides[k + 1][1]
                cur_lines = [l for l in sr.signature if l]
                if (sr.title == nxt.title and cur_lines and len(nxt.signature) > len(sr.signature)
                        and all(l in nxt.signature for l in cur_lines)
                        and all(any(iou(f["rect"], g["rect"]) >= 0.8 for g in nxt.figures) for f in sr.figures)):
                    dropped.append(sr.number)
                    continue
            keep.append((page, sr))
        slides = keep

    title = (doc.metadata or {}).get("title", "").strip()
    if not title:
        title = pdf_path.stem.replace("_", " ").strip()

    labels = [ctx["page_labels"][k] for k in sorted(ctx["page_labels"])]
    page_labels = ""
    if labels:
        page_labels = f"{labels[0]}-{labels[-1]}" if len(labels) > 1 else labels[0]

    md = []
    dpi = opts.get("figure_dpi", 200)
    for page, sr in slides:
        heading = f"## Slide {sr.number}" + (f": {sr.title}" if sr.title else "")
        md += [heading, ""]
        body = blocks_to_md(sr.body)
        if body:
            md += [body, ""]
        for k, fig in enumerate(sr.figures, 1):
            fname = f"{name}_slide{sr.number:02d}_fig{k}.png"
            clip = (fig["rect"] + (-3, -3, 3, 3)) & page.rect
            page.get_pixmap(dpi=dpi, clip=clip, alpha=False).save(img_dir / fname)
            stats.figures += 1
            md += [f"![Slide {sr.number}, figure {k}](images/{fname})",
                   f"*Figure {k}: [description placeholder]*"]
            if fig["labels"]:
                md.append("Figure labels: " + " · ".join(fig["labels"]))
            if fig["ocr"]:
                md += ["", "> Text in the image (OCR):"] + [f"> {l}" for l in fig["ocr"]]
            md.append("")
        if sr.links:
            md += ["Links: " + ", ".join(f"[{t}]({u})" for t, u in sr.links), ""]

    text = re.sub(r"\n{3,}", "\n\n", "\n".join(md)).strip() + "\n"
    report = {"file": pdf_path.name, "title": title, "pages": len(doc), "slides": len(slides),
              "page_labels": page_labels, "dropped_build_slides": dropped, "figures": stats.figures,
              "tables": stats.tables, "formulas_inline": stats.formulas_inline,
              "formulas_display": stats.formulas_display, "complex_formulas": stats.complex_formulas,
              "pix2tex_used": stats.pix2tex_used, "pix2tex_rejected": stats.pix2tex_rejected,
              "ocr_image_texts": stats.ocr_image_texts, "warnings": stats.warnings, "math_log": stats.math_log}
    doc.close()
    return text, report


def write_combined(parts, out_dir, name):
    out_dir = Path(out_dir)
    reports = [report for _, report in parts]
    titles = [r["title"] for r in reports]
    heading = titles[0] if titles and all(t == titles[0] for t in titles) else name.replace("_", " ")
    fm = {"title": heading, "source_files": [r["file"] for r in reports],
          "pages": sum(r["pages"] for r in reports), "slides": sum(r["slides"] for r in reports),
          "converted": datetime.date.today().isoformat()}
    labels = [r["page_labels"] for r in reports if r.get("page_labels")]
    if labels:
        fm["page_labels"] = ", ".join(labels)
    dropped = [f"{r['file']}: {', '.join(map(str, r['dropped_build_slides']))}"
               for r in reports if r.get("dropped_build_slides")]
    if dropped:
        fm["dropped_build_slides"] = "; ".join(dropped)
    sections = [text.strip() for text, _ in parts if text.strip()]
    md = ["---"] + [f"{k}: {json.dumps(v, ensure_ascii=False)}" for k, v in fm.items()] + ["---", "", f"# {heading}", ""]
    text = re.sub(r"\n{3,}", "\n\n", ("\n".join(md) + "\n" + "\n\n".join(sections)).strip()) + "\n"
    path = out_dir / f"{name}.md"
    path.write_text(text, encoding="utf-8")
    img_dir = out_dir / "images"
    if img_dir.is_dir() and not any(img_dir.iterdir()):
        img_dir.rmdir()
    return path

## 5. Upload PDFs

Select one or more `.pdf` files, or a `.zip` that contains them. If `DRIVE_INPUT_FOLDER` is set, Drive is mounted and the PDFs are read from that folder instead.

In [ ]:
#@title Upload (or read from Drive)
import os, shutil, zipfile
from pathlib import Path

INPUT_DIR = Path("/content/input")
OUTPUT_DIR = Path("/content/slides_markdown")
for d in (INPUT_DIR, OUTPUT_DIR):
    shutil.rmtree(d, ignore_errors=True)
    d.mkdir(parents=True)


def add_pdf(name, data):
    target = INPUT_DIR / Path(name).name
    stem, n = target.stem, 2
    while target.exists():
        target = INPUT_DIR / f"{stem}_{n}.pdf"
        n += 1
    target.write_bytes(data)


if DRIVE_INPUT_FOLDER or DRIVE_OUTPUT_FOLDER:
    from google.colab import drive
    drive.mount("/content/drive")

if DRIVE_INPUT_FOLDER:
    for p in sorted(Path(DRIVE_INPUT_FOLDER).glob("*.pdf")):
        add_pdf(p.name, p.read_bytes())
else:
    from google.colab import files
    for name, data in files.upload().items():
        if name.lower().endswith(".zip"):
            zip_path = INPUT_DIR / "_upload.zip"
            zip_path.write_bytes(data)
            with zipfile.ZipFile(zip_path) as z:
                for info in z.infolist():
                    base = Path(info.filename).name
                    if base.lower().endswith(".pdf") and not info.filename.startswith("__MACOSX") and not base.startswith("."):
                        add_pdf(base, z.read(info))
            zip_path.unlink()
        elif name.lower().endswith(".pdf"):
            add_pdf(name, data)
        else:
            print("Skipped (not a PDF or ZIP):", name)

PDFS = sorted(INPUT_DIR.glob("*.pdf"))
print(f"{len(PDFS)} PDF(s):")
for p in PDFS:
    print(" -", p.name)

## 6. Convert

In [ ]:
#@title Convert all PDFs
import time
REPORTS = []
SECTIONS = []
for pdf in PDFS:
    t0 = time.time()
    try:
        text, rep = convert_pdf(pdf, OUTPUT_DIR, OPTS, latex_ocr)
    except Exception as e:
        print(f"FAILED  {pdf.name}: {type(e).__name__}: {e}")
        continue
    rep["seconds"] = round(time.time() - t0, 1)
    REPORTS.append(rep)
    SECTIONS.append(text)
    dropped = f", dropped build slides {rep['dropped_build_slides']}" if rep["dropped_build_slides"] else ""
    print(f"OK      {rep['file']}  ({rep['seconds']} s)")
    print(f"        {rep['pages']} pages -> {rep['slides']} slides{dropped}")
    print(f"        {rep['figures']} figures, {rep['tables']} tables, "
          f"{rep['formulas_inline']} inline + {rep['formulas_display']} display formulas, "
          f"pix2tex used {rep['pix2tex_used']} / rejected {rep['pix2tex_rejected']}, "
          f"{rep['ocr_image_texts']} image texts from OCR")
    for w in rep["warnings"]:
        print("        warning:", w)

OUTPUT_FILE = None
if REPORTS:
    name = shared_output_name([Path(r["file"]).stem for r in REPORTS], OUTPUT_NAME)
    OUTPUT_FILE = write_combined(list(zip(SECTIONS, REPORTS)), OUTPUT_DIR, name)
    for r in REPORTS:
        r["output"] = OUTPUT_FILE.name
    print(f"Wrote {OUTPUT_FILE.name} and images/")

(OUTPUT_DIR / "conversion_report.json").write_text(
    json.dumps([{k: v for k, v in r.items() if k != "math_log"} for r in REPORTS], indent=2, ensure_ascii=False),
    encoding="utf-8")

## 7. Preview and download

In [ ]:
#@title Preview the Markdown file
PREVIEW_LINES = 80  #@param {type:"integer"}
if OUTPUT_FILE:
    text = OUTPUT_FILE.read_text(encoding="utf-8")
    print(f"--- {OUTPUT_FILE.name} ---")
    print("\n".join(text.splitlines()[:PREVIEW_LINES]))

In [ ]:
#@title Download everything as one ZIP
archive = shutil.make_archive("/content/slides_markdown", "zip", OUTPUT_DIR)
print("Created", archive)
if DRIVE_OUTPUT_FOLDER:
    dest = Path(DRIVE_OUTPUT_FOLDER)
    dest.mkdir(parents=True, exist_ok=True)
    if OUTPUT_FILE:
        shutil.copy2(OUTPUT_FILE, dest / OUTPUT_FILE.name)
    images = OUTPUT_DIR / "images"
    if images.is_dir():
        shutil.copytree(images, dest / "images", dirs_exist_ok=True)
    print("Copied to", dest)
from google.colab import files
files.download(archive)